<a href="https://colab.research.google.com/github/AI4Finance-Foundation/FinRL/blob/master/tutorials/2-Advance/FinRL_PortfolioAllocation_Explainable_DRL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
import datetime

%matplotlib inline
from finrl.config_tickers import DOW_30_TICKER
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent,DRLEnsembleAgent
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline

C:\Users\perfe\AppData\Roaming\Python\Python39\site-packages\pandas_datareader\compat\__init__.py:11: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  PANDAS_VERSION = LooseVersion(pd.__version__)
C:\Users\perfe\.conda\envs\finrl\lib\site-packages\pyfolio\pos.py:26: UserWarning: Module "zipline.assets" not found; mutltipliers will not be applied to position notionals.
  warnings.warn(
C:\Users\perfe\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\multiclass.py:14: DeprecationWarning: Please import `spmatrix` from the `scipy.sparse` namespace; the `scipy.sparse.base` namespace is deprecated and will be removed in SciPy 2.0.0.
  from scipy.sparse.base import spmatrix
C:\Users\perfe\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\optimize.py:18: DeprecationWarning: `scipy.optimize.linesearch.line_search_wolfe2` is deprecated along with the `scipy.optimize.linesearch` namespace. `scipy.optimize.linesearch.line_search_wol

In [3]:
df=pd.read_csv('*********trade20241231.csv')

In [4]:
fe = FeatureEngineer(
                    use_technical_indicator=True,
                    use_turbulence=False,
                    user_defined_feature = False)

df = fe.preprocess_data(df)

Successfully added technical indicators


## Add covariance matrix as states

In [5]:
# add covariance matrix as states
df=df.sort_values(['date','tic'],ignore_index=True)
df.index = df.date.factorize()[0]

cov_list = []
return_list = []

# look back is one year
lookback=252
for i in range(lookback,len(df.index.unique())):
  data_lookback = df.loc[i-lookback:i,:]
  price_lookback=data_lookback.pivot_table(index = 'date',columns = 'tic', values = 'close')
  return_lookback = price_lookback.pct_change().dropna()
  return_list.append(return_lookback)

  covs = return_lookback.cov().values 
  cov_list.append(covs)

  
df_cov = pd.DataFrame({'date':df.date.unique()[lookback:],'cov_list':cov_list,'return_list':return_list})
df = df.merge(df_cov, on='date')
df = df.sort_values(['date','tic']).reset_index(drop=True)

In [6]:
df

,tic,date,high,low,close,volume,TTM,PBR,PS,PCF,...,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,cov_list,return_list
0,ABC,2019/1/22,3.59,3.56,3.57,485998433,6.1078,0.8058,2.0873,-54.7880,...,-0.026887,3.930377,3.348623,48.684346,-48.342541,5.415330,3.634000,3.590333,"[[0.0012892994720226584, 0.0009882654827793387...",tic ABC BOC BOCOM ...
1,BOC,2019/1/22,3.58,3.55,3.56,376696803,5.8167,0.7086,2.0790,1.4359,...,-0.004310,3.711014,3.431986,49.300442,8.550118,23.458672,3.559667,3.548333,"[[0.0012892994720226584, 0.0009882654827793387...",tic ABC BOC BOCOM ...
2,BOCOM,2019/1/22,6.05,6.00,6.02,280710307,6.1151,0.7181,2.1023,30.6774,...,0.082618,6.108388,5.518612,55.906955,135.097493,15.153585,5.735333,5.670333,"[[0.0012892994720226584, 0.0009882654827793387...",tic ABC BOC BOCOM ...
3,CCB,2019/1/22,6.70,6.56,6.60,322753931,6.4676,0.8956,2.5043,5.7024,...,-0.085270,7.358410,6.047590,48.138755,-23.765996,5.823744,6.698333,6.719667,"[[0.0012892994720226584, 0.0009882654827793387...",tic ABC BOC BOCOM ...
4,CEB,2019/1/22,3.97,3.86,3.89,243405728,6.0662,0.7315,1.8521,5.1358,...,0.038709,4.036425,3.651575,52.955818,70.830485,7.843137,3.816000,3.702000,"[[0.0012892994720226584, 0.0009882654827793387...",tic ABC BOC BOCOM ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21700,HXB,2024/9/9,6.59,6.44,6.52,403933616,3.8850,0.3631,1.1133,7.1984,...,0.086512,7.267561,5.744439,50.515819,31.541101,12.964378,6.416000,6.395167,"[[0.001069862202249902, 0.0008281905367535538,...",tic ABC BOC BOCOM ...
21701,ICBC,2024/9/9,5.65,5.51,5.61,1576623851,5.5430,0.5730,2.3716,2.4122,...,-0.060115,6.205240,5.361760,46.630266,-86.834992,5.255048,5.904667,5.958333,"[[0.001069862202249902, 0.0008281905367535538,...",tic ABC BOC BOCOM ...
21702,PAD,2024/9/9,10.07,9.84,9.85,1624857668,4.0716,0.4640,1.1606,2.5187,...,0.051456,11.524945,8.916055,47.507398,-55.120018,7.779079,10.209000,10.199167,"[[0.001069862202249902, 0.0008281905367535538,...",tic ABC BOC BOCOM ...
21703,PSBC,2024/9/9,4.72,4.61,4.63,737664406,5.3685,0.5703,1.3405,2.1530,...,-0.028038,5.181224,4.331776,46.867669,-57.901135,7.885267,4.809333,4.892667,"[[0.001069862202249902, 0.0008281905367535538,...",tic ABC BOC BOCOM ...


In [7]:
train = data_split(df, '2018-01-02','2022-12-31')


## Environment for Portfolio Allocation


In [8]:
import numpy as np
import pandas as pd
from gym.utils import seeding
import gym
from gym import spaces
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from stable_baselines3.common.vec_env import DummyVecEnv


class StockPortfolioEnv(gym.Env):
    """A portfolio allocation environment for OpenAI gym

    Attributes
    ----------
        df: DataFrame
            input data
        stock_dim : int
            number of unique stocks
        hmax : int
            maximum number of shares to trade
        initial_amount : int
            start money
        transaction_cost_pct: float
            transaction cost percentage per trade
        reward_scaling: float
            scaling factor for reward, good for training
        state_space: int
            the dimension of input features
        action_space: int
            equals stock dimension
        tech_indicator_list: list
            a list of technical indicator names
        turbulence_threshold: int
            a threshold to control risk aversion
        day: int
            an increment number to control date

    Methods
    -------
    _sell_stock()
        perform sell action based on the sign of the action
    _buy_stock()
        perform buy action based on the sign of the action
    step()
        at each step the agent will return actions, then 
        we will calculate the reward, and return the next observation.
    reset()
        reset the environment
    render()
        use render to return other functions
    save_asset_memory()
        return account value at each time step
    save_action_memory()
        return actions/positions at each time step
        

    """
    metadata = {'render.modes': ['human']}

    def __init__(self, 
                df,
                stock_dim,
                hmax,
                initial_amount,
                transaction_cost_pct,
                reward_scaling,
                state_space,
                action_space,
                tech_indicator_list,
                turbulence_threshold=None,
                lookback=252,
                day = 0):
        #super(StockEnv, self).__init__()
        #money = 10 , scope = 1
        self.day = day
        self.lookback=lookback
        self.df = df
        self.stock_dim = stock_dim
        self.hmax = hmax
        self.initial_amount = initial_amount
        self.transaction_cost_pct =transaction_cost_pct
        self.reward_scaling = reward_scaling
        self.state_space = state_space
        self.action_space = action_space
        self.tech_indicator_list = tech_indicator_list

        # action_space normalization and shape is self.stock_dim
        self.action_space = spaces.Box(low = 0, high = 1,shape = (self.action_space,)) 
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape = (self.state_space+len(self.tech_indicator_list),self.state_space))

        # load data from a pandas dataframe
        self.data = self.df.loc[self.day,:]
        self.covs = self.data['cov_list'].values[0]
        self.state =  np.append(np.array(self.covs), [self.data[tech].values.tolist() for tech in self.tech_indicator_list ], axis=0)
        self.terminal = False     
        self.turbulence_threshold = turbulence_threshold        
        # initalize state: inital portfolio return + individual stock return + individual weights
        self.portfolio_value = self.initial_amount

        # memorize portfolio value each step
        self.asset_memory = [self.initial_amount]
        # memorize portfolio return each step
        self.portfolio_return_memory = [0]
        self.actions_memory=[[1/self.stock_dim]*self.stock_dim]
        self.date_memory=[self.data.date.unique()[0]]

        
    def step(self, actions):
        self.terminal = self.day >= len(self.df.index.unique())-1

        if self.terminal:
            df = pd.DataFrame(self.portfolio_return_memory)
            df.columns = ['daily_return']
            plt.plot(df.daily_return.cumsum(),'r')
            plt.savefig('results/cumulative_reward.png')
            plt.close()
            
            plt.plot(self.portfolio_return_memory,'r')
            plt.savefig('results/rewards.png')
            plt.close()

            print("=================================")
            print("begin_total_asset:{}".format(self.asset_memory[0]))           
            print("end_total_asset:{}".format(self.portfolio_value))

            df_daily_return = pd.DataFrame(self.portfolio_return_memory)
            df_daily_return.columns = ['daily_return']
            if df_daily_return['daily_return'].std() !=0:
              sharpe = (252**0.5)*df_daily_return['daily_return'].mean()/ \
                       df_daily_return['daily_return'].std()
              print("Sharpe: ",sharpe)
            print("=================================")
            
            return self.state, self.reward, self.terminal,{}

        else:
            weights = self.softmax_normalization(actions) 
            self.actions_memory.append(weights)
            last_day_memory = self.data

            #load next state
            self.day += 1
            self.data = self.df.loc[self.day,:]
            self.covs = self.data['cov_list'].values[0]
            self.state =  np.append(np.array(self.covs), [self.data[tech].values.tolist() for tech in self.tech_indicator_list ], axis=0)
            portfolio_return = sum(((self.data.close.values / last_day_memory.close.values)-1)*weights)
            log_portfolio_return = np.log(sum((self.data.close.values / last_day_memory.close.values)*weights))
            # update portfolio value
            new_portfolio_value = self.portfolio_value*(1+portfolio_return)
            self.portfolio_value = new_portfolio_value

            # save into memory
            self.portfolio_return_memory.append(portfolio_return)
            self.date_memory.append(self.data.date.unique()[0])            
            self.asset_memory.append(new_portfolio_value)

            # the reward is the new portfolio value or end portfolo value
            self.reward = new_portfolio_value
            

        return self.state, self.reward, self.terminal, {}

    def reset(self):
        self.asset_memory = [self.initial_amount]
        self.day = 0
        self.data = self.df.loc[self.day,:]
        # load states
        self.covs = self.data['cov_list'].values[0]
        self.state =  np.append(np.array(self.covs), [self.data[tech].values.tolist() for tech in self.tech_indicator_list ], axis=0)
        self.portfolio_value = self.initial_amount
        #self.cost = 0
        #self.trades = 0
        self.terminal = False 
        self.portfolio_return_memory = [0]
        self.actions_memory=[[1/self.stock_dim]*self.stock_dim]
        self.date_memory=[self.data.date.unique()[0]] 
        return self.state
    
    def render(self, mode='human'):
        return self.state
        
    def softmax_normalization(self, actions):
        numerator = np.exp(actions)
        denominator = np.sum(np.exp(actions))
        softmax_output = numerator/denominator
        return softmax_output

    
    def save_asset_memory(self):
        date_list = self.date_memory
        portfolio_return = self.portfolio_return_memory
        #print(len(date_list))
        #print(len(asset_list))
        df_account_value = pd.DataFrame({'date':date_list,'daily_return':portfolio_return})
        return df_account_value

    def save_action_memory(self):
        # date and close price length must match actions length
        date_list = self.date_memory
        df_date = pd.DataFrame(date_list)
        df_date.columns = ['date']
        
        action_list = self.actions_memory
        df_actions = pd.DataFrame(action_list)
        df_actions.columns = self.data.tic.values
        df_actions.index = df_date.date
        #df_actions = pd.DataFrame({'date':date_list,'actions':action_list})
        return df_actions

    def _seed(self, seed=None):
        self.np_random, seed = seeding.np_random(seed)
        return [seed]

    def get_sb_env(self):
        e = DummyVecEnv([lambda: self])
        obs = e.reset()
        return e, obs

In [9]:
stock_dimension = len(train.tic.unique())
state_space = stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")
tech_indicator_list = ['macd', 'rsi_30', 'cci_30', 'dx_30']
feature_dimension = len(tech_indicator_list)
print(f"Feature Dimension: {feature_dimension}")

Stock Dimension: 15, State Space: 15
Feature Dimension: 4


In [10]:
env_kwargs = {
    "hmax": 100, 
    "initial_amount": 1000000, 
    "transaction_cost_pct": 0, 
    "state_space": state_space, 
    "stock_dim": stock_dimension, 
    "tech_indicator_list": tech_indicator_list, 
    "action_space": stock_dimension, 
    "reward_scaling": 1e-1
    
}

e_train_gym = StockPortfolioEnv(df = train, **env_kwargs)

In [11]:
env_train, _ = e_train_gym.get_sb_env()
print(type(env_train))

C:\Users\perfe\.conda\envs\finrl\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


<class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>


### Model 1: **A2C**


In [12]:
agent = DRLAgent(env = env_train)

A2C_PARAMS = {"n_steps": 10, "ent_coef": 0.005, "learning_rate": 0.0004}
model_a2c = agent.get_model(model_name="a2c",model_kwargs = A2C_PARAMS)

{'n_steps': 10, 'ent_coef': 0.005, 'learning_rate': 0.0004}
Using cuda device


C:\Users\perfe\.conda\envs\finrl\lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [13]:
trained_a2c = agent.train_model(model=model_a2c, tb_log_name='a2c',
                                total_timesteps=40000)

begin_total_asset:1000000
end_total_asset:1029566.3794942652
Sharpe:  0.2029766269379108
------------------------------------
| time/                 |          |
|    fps                | 218      |
|    iterations         | 100      |
|    time_elapsed       | 4        |
|    total_timesteps    | 1000     |
| train/                |          |
|    entropy_loss       | -21.3    |
|    explained_variance | 1.79e-07 |
|    learning_rate      | 0.0004   |
|    n_updates          | 99       |
|    policy_loss        | 1.15e+08 |
|    reward             | 986612.0 |
|    std                | 1        |
|    value_loss         | 3.25e+13 |
------------------------------------
begin_total_asset:1000000
end_total_asset:1060042.105023854
Sharpe:  0.2310428324276817
-------------------------------------
| time/                 |           |
|    fps                | 219       |
|    iterations         | 200       |
|    time_elapsed       | 9         |
|    total_timesteps    | 2000      |
| t

### Model 2: **PPO**


In [14]:
agent = DRLAgent(env = env_train)
PPO_PARAMS = {
    "n_steps": 2048,
    "ent_coef": 0.005,
    "learning_rate": 0.001,
    "batch_size": 128,
}
model_ppo = agent.get_model("ppo",model_kwargs = PPO_PARAMS)

{'n_steps': 2048, 'ent_coef': 0.005, 'learning_rate': 0.001, 'batch_size': 128}
Using cuda device


C:\Users\perfe\.conda\envs\finrl\lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [15]:
trained_ppo = agent.train_model(model=model_ppo, 
                             tb_log_name='ppo',
                             total_timesteps=40000)

begin_total_asset:1000000
end_total_asset:1154549.3725893972
Sharpe:  0.31651930366137654
begin_total_asset:1000000
end_total_asset:1002094.1924291159
Sharpe:  0.17705310563816776
----------------------------------
| time/              |           |
|    fps             | 232       |
|    iterations      | 1         |
|    time_elapsed    | 8         |
|    total_timesteps | 2048      |
| train/             |           |
|    reward          | 1161886.4 |
----------------------------------
begin_total_asset:1000000
end_total_asset:1073699.7753258396
Sharpe:  0.24347874948943016
begin_total_asset:1000000
end_total_asset:1119974.8770117557
Sharpe:  0.2854305611558034
begin_total_asset:1000000
end_total_asset:1135379.7915238128
Sharpe:  0.2996655856801038
-------------------------------------------
| time/                   |               |
|    fps                  | 205           |
|    iterations           | 2             |
|    time_elapsed         | 19            |
|    total_timest

## Back-Testing
Assume that we have $1,000,000 initial capital at 2020-01-01. We use the PPO, A2C, SVM, Linear Regression, Decision Tree, Random Foreset models to trade Dow jones 30 constituent stocks.


In [16]:
trade = data_split(df,'2022-01-01', '2024-12-31')
e_trade_gym = StockPortfolioEnv(df = trade, **env_kwargs)


In [17]:
import torch
%matplotlib inline
import plotly.express as px


In [18]:
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
import pandas as pd
from pypfopt import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns
from pypfopt import objective_functions
unique_tic = trade.tic.unique()
unique_trade_date = trade.date.unique()

In [20]:
agent = DRLAgent(env = env_train)
model_ddpg = agent.get_model("ddpg")

{'batch_size': 128, 'buffer_size': 50000, 'learning_rate': 0.001}
Using cuda device


In [21]:
trained_ddpg = agent.train_model(model=model_ddpg, 
                             tb_log_name='ddpg',
                             total_timesteps=50000)

begin_total_asset:1000000
end_total_asset:1066238.6819305208
Sharpe:  0.23595792164955715
begin_total_asset:1000000
end_total_asset:1070831.2605075026
Sharpe:  0.2404182132423641
begin_total_asset:1000000
end_total_asset:1070831.2605075026
Sharpe:  0.2404182132423641
begin_total_asset:1000000
end_total_asset:1070831.2605075026
Sharpe:  0.2404182132423641
----------------------------------
| time/              |           |
|    episodes        | 4         |
|    fps             | 73        |
|    time_elapsed    | 39        |
|    total_timesteps | 2884      |
| train/             |           |
|    actor_loss      | -1.28e+07 |
|    critic_loss     | 4.2e+10   |
|    learning_rate   | 0.001     |
|    n_updates       | 2783      |
|    reward          | 1070831.2 |
----------------------------------
begin_total_asset:1000000
end_total_asset:1070831.2605075026
Sharpe:  0.2404182132423641
begin_total_asset:1000000
end_total_asset:1070831.2605075026
Sharpe:  0.2404182132423641
begin_tota

In [22]:
agent = DRLAgent(env = env_train)
TD3_PARAMS = {"batch_size": 100, 
              "buffer_size": 1000000, 
              "learning_rate": 0.001}

model_td3 = agent.get_model("td3",model_kwargs = TD3_PARAMS)

{'batch_size': 100, 'buffer_size': 1000000, 'learning_rate': 0.001}
Using cuda device


In [23]:
trained_td3 = agent.train_model(model=model_td3, 
                             tb_log_name='td3',
                             total_timesteps=30000)

begin_total_asset:1000000
end_total_asset:1139215.8227675476
Sharpe:  0.30321449769740827
begin_total_asset:1000000
end_total_asset:1116750.3285977193
Sharpe:  0.28375092651538975
begin_total_asset:1000000
end_total_asset:1116750.3285977193
Sharpe:  0.28375092651538975
begin_total_asset:1000000
end_total_asset:1116750.3285977193
Sharpe:  0.28375092651538975
----------------------------------
| time/              |           |
|    episodes        | 4         |
|    fps             | 79        |
|    time_elapsed    | 36        |
|    total_timesteps | 2884      |
| train/             |           |
|    actor_loss      | -6.72e+06 |
|    critic_loss     | 6.17e+11  |
|    learning_rate   | 0.001     |
|    n_updates       | 2783      |
|    reward          | 1116750.4 |
----------------------------------
begin_total_asset:1000000
end_total_asset:1116750.3285977193
Sharpe:  0.28375092651538975
begin_total_asset:1000000
end_total_asset:1116750.3285977193
Sharpe:  0.28375092651538975
begin

In [24]:
agent = DRLAgent(env = env_train)
SAC_PARAMS = {
    "batch_size": 128,
    "buffer_size": 1000000,
    "learning_rate": 0.0001,
    "learning_starts": 100,
    "ent_coef": "auto_0.1",
}

model_sac = agent.get_model("sac",model_kwargs = SAC_PARAMS)

{'batch_size': 128, 'buffer_size': 1000000, 'learning_rate': 0.0001, 'learning_starts': 100, 'ent_coef': 'auto_0.1'}
Using cuda device


In [25]:
trained_sac = agent.train_model(model=model_sac, 
                             tb_log_name='sac',
                             total_timesteps=60000)

begin_total_asset:1000000
end_total_asset:1083278.5201740407
Sharpe:  0.25047927427566047
begin_total_asset:1000000
end_total_asset:1032899.8496915739
Sharpe:  0.19781350170234396
begin_total_asset:1000000
end_total_asset:1032754.209812141
Sharpe:  0.19765900232337788
begin_total_asset:1000000
end_total_asset:1032657.5346613653
Sharpe:  0.19755875778247126
-----------------------------------
| time/              |            |
|    episodes        | 4          |
|    fps             | 42         |
|    time_elapsed    | 68         |
|    total_timesteps | 2884       |
| train/             |            |
|    actor_loss      | -3.42e+06  |
|    critic_loss     | 5.7e+11    |
|    ent_coef        | 0.138      |
|    ent_coef_loss   | 252        |
|    learning_rate   | 0.0001     |
|    n_updates       | 2783       |
|    reward          | 1032657.56 |
-----------------------------------
begin_total_asset:1000000
end_total_asset:1032502.689696592
Sharpe:  0.19739973273474395
begin_total_

In [26]:
from pyfolio import timeseries

df_daily_return_a2c, df_actions_a2c = DRLAgent.DRL_prediction(model=trained_a2c,
                        environment = e_trade_gym)
df_daily_return_ppo, df_actions_ppo = DRLAgent.DRL_prediction(model=trained_ppo,
                        environment = e_trade_gym)
time_ind = pd.Series(df_daily_return_a2c.date)
a2c_cumpod =(df_daily_return_a2c.daily_return+1).cumprod()-1
ppo_cumpod =(df_daily_return_ppo.daily_return+1).cumprod()-1
df_daily_return_ddpg, df_actions_ddpg = DRLAgent.DRL_prediction(model=trained_ddpg,
                        environment = e_trade_gym)
df_daily_return_td3, df_actions_td3 = DRLAgent.DRL_prediction(model=trained_td3,
                        environment = e_trade_gym)
df_daily_return_sac, df_actions_sac = DRLAgent.DRL_prediction(model=trained_sac,
                        environment = e_trade_gym)
ddpg_cumpod =(df_daily_return_ddpg.daily_return+1).cumprod()-1
td3_cumpod =(df_daily_return_td3.daily_return+1).cumprod()-1
sac_cumpod =(df_daily_return_sac.daily_return+1).cumprod()-1
#DRL_strat_a2c = convert_daily_return_to_pyfolio_ts(df_daily_return_a2c)
#DRL_strat_ppo = convert_daily_return_to_pyfolio_ts(df_daily_return_ppo)

#perf_func = timeseries.perf_stats 
#perf_stats_all_a2c = perf_func( returns=DRL_strat_a2c, 
                             # factor_returns=DRL_strat_a2c, 
                               # positions=None, transactions=None, turnover_denom="AGB")
#perf_stats_all_ppo = perf_func( returns=DRL_strat_ppo, 
                              #factor_returns=DRL_strat_ppo, 
                               # positions=None, transactions=None, turnover_denom="AGB")

C:\Users\perfe\.conda\envs\finrl\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


begin_total_asset:1000000
end_total_asset:996319.4055760088
Sharpe:  0.15861616892948663
hit end!


C:\Users\perfe\.conda\envs\finrl\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


begin_total_asset:1000000
end_total_asset:1016629.7864356985
Sharpe:  0.19251637144247524
hit end!


C:\Users\perfe\.conda\envs\finrl\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


begin_total_asset:1000000
end_total_asset:1002657.7742074706
Sharpe:  0.16991358130872125
hit end!


C:\Users\perfe\.conda\envs\finrl\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


begin_total_asset:1000000
end_total_asset:993509.9188861985
Sharpe:  0.15595309588938303
hit end!


C:\Users\perfe\.conda\envs\finrl\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


begin_total_asset:1000000
end_total_asset:1005326.113884023
Sharpe:  0.16878782320575772
hit end!


In [27]:


# 计算年化收益率
def annualized_return(df):
    total_return = df['daily_return'].sum()  # 总收益
    annual_return = (1 + total_return) ** (252 / len(df)) - 1  # 252 是年交易日数
    return annual_return

# 计算最大回撤
def max_drawdown(df):
    cumulative_returns = (1 + df['daily_return']).cumprod()
    drawdowns = cumulative_returns / cumulative_returns.cummax() - 1
    max_drawdown = drawdowns.min()
    return max_drawdown

# 计算年化波动率
def annualized_volatility(df):
    daily_volatility = df['daily_return'].std()  # 日波动率
    annual_volatility = daily_volatility * np.sqrt(252)  # 转换为年波动率
    return annual_volatility

# 计算夏普比率
def sharpe_ratio(df, risk_free_rate=0):
    ann_return = annualized_return(df)
    ann_volatility = annualized_volatility(df)
    return (ann_return - risk_free_rate) / ann_volatility

# 计算欧米伽比率
def omega_ratio(df, threshold=0):
    positive_returns = df[df['daily_return'] > threshold]['daily_return'].sum()
    negative_returns = abs(df[df['daily_return'] < threshold]['daily_return'].sum())
    if negative_returns == 0:
        return np.inf  # 如果没有负回报，欧米伽比率为无穷大
    return positive_returns / negative_returns



In [28]:
df_daily_return_sac.set_index('date', inplace=True)
df_daily_return_sac['date'] = pd.date_range(start='2022-01-01', periods=len(df_daily_return_sac), freq='D')

# 设置日期为索引
df_daily_return_sac.set_index('date', inplace=True)

# 计算累积收益率 -> 假设初始账户值为 1
df_daily_return_sac['account_value'] = (1 + df_daily_return_sac['daily_return']).cumprod()
ann_return = annualized_return(df_daily_return_sac)
max_dd = max_drawdown(df_daily_return_sac)
sharpe = sharpe_ratio(df_daily_return_sac)
omega = omega_ratio(df_daily_return_sac)

# 输出结果
print(f"年化收益率: {ann_return * 100:.2f}%")
print(f"最大回撤: {max_dd * 100:.2f}%")
print(f"夏普比率: {sharpe:.2f}")
print(f"欧米伽比率: {omega:.2f}")

年化收益率: 5.30%
最大回撤: -20.33%
夏普比率: 0.16
欧米伽比率: 1.04


In [29]:
df_daily_return_ppo.set_index('date', inplace=True)
df_daily_return_ppo['date'] = pd.date_range(start='2022-01-01', periods=len(df_daily_return_ppo), freq='D')

# 设置日期为索引
df_daily_return_ppo.set_index('date', inplace=True)

# 计算累积收益率 -> 假设初始账户值为 1
df_daily_return_ppo['account_value'] = (1 + df_daily_return_ppo['daily_return']).cumprod()
ann_return = annualized_return(df_daily_return_ppo)
max_dd = max_drawdown(df_daily_return_ppo)
sharpe = sharpe_ratio(df_daily_return_ppo)
omega = omega_ratio(df_daily_return_ppo)

# 输出结果
print(f"年化收益率: {ann_return * 100:.2f}%")
print(f"最大回撤: {max_dd * 100:.2f}%")
print(f"夏普比率: {sharpe:.2f}")
print(f"欧米伽比率: {omega:.2f}")

年化收益率: 6.26%
最大回撤: -21.06%
夏普比率: 0.19
欧米伽比率: 1.04


In [30]:
df_daily_return_a2c.set_index('date', inplace=True)
df_daily_return_a2c['date'] = pd.date_range(start='2022-01-01', periods=len(df_daily_return_a2c), freq='D')

# 设置日期为索引
df_daily_return_a2c.set_index('date', inplace=True)

# 计算累积收益率 -> 假设初始账户值为 1
df_daily_return_a2c['account_value'] = (1 + df_daily_return_a2c['daily_return']).cumprod()
ann_return = annualized_return(df_daily_return_a2c)
max_dd = max_drawdown(df_daily_return_a2c)
sharpe = sharpe_ratio(df_daily_return_a2c)
omega = omega_ratio(df_daily_return_a2c)

# 输出结果
print(f"年化收益率: {ann_return * 100:.2f}%")
print(f"最大回撤: {max_dd * 100:.2f}%")
print(f"夏普比率: {sharpe:.2f}")
print(f"欧米伽比率: {omega:.2f}")

年化收益率: 5.12%
最大回撤: -20.75%
夏普比率: 0.15
欧米伽比率: 1.04


In [31]:
df_daily_return_td3.set_index('date', inplace=True)
df_daily_return_td3['date'] = pd.date_range(start='2022-01-01', periods=len(df_daily_return_td3), freq='D')

# 设置日期为索引
df_daily_return_td3.set_index('date', inplace=True)

# 计算累积收益率 -> 假设初始账户值为 1
df_daily_return_td3['account_value'] = (1 + df_daily_return_td3['daily_return']).cumprod()
ann_return = annualized_return(df_daily_return_td3)
max_dd = max_drawdown(df_daily_return_td3)
sharpe = sharpe_ratio(df_daily_return_td3)
omega = omega_ratio(df_daily_return_td3)

# 输出结果
print(f"年化收益率: {ann_return * 100:.2f}%")
print(f"最大回撤: {max_dd * 100:.2f}%")
print(f"夏普比率: {sharpe:.2f}")
print(f"欧米伽比率: {omega:.2f}")

年化收益率: 5.07%
最大回撤: -21.63%
夏普比率: 0.15
欧米伽比率: 1.04


In [32]:
df_daily_return_ddpg.set_index('date', inplace=True)
df_daily_return_ddpg['date'] = pd.date_range(start='2022-01-01', periods=len(df_daily_return_ddpg), freq='D')

# 设置日期为索引
df_daily_return_ddpg.set_index('date', inplace=True)

# 计算累积收益率 -> 假设初始账户值为 1
df_daily_return_ddpg['account_value'] = (1 + df_daily_return_ddpg['daily_return']).cumprod()
ann_return = annualized_return(df_daily_return_ddpg)
max_dd = max_drawdown(df_daily_return_ddpg)
sharpe = sharpe_ratio(df_daily_return_ddpg)
omega = omega_ratio(df_daily_return_ddpg)

# 输出结果
print(f"年化收益率: {ann_return * 100:.2f}%")
print(f"最大回撤: {max_dd * 100:.2f}%")
print(f"夏普比率: {sharpe:.2f}")
print(f"欧米伽比率: {omega:.2f}")

年化收益率: 5.51%
最大回撤: -20.63%
夏普比率: 0.17
欧米伽比率: 1.04


In [33]:
df_daily_return_a2c.to_csv('******trade20241231a2c.csv')
df_daily_return_ppo.to_csv('******trade20241231ppo.csv')
df_daily_return_ddpg.to_csv('******trade20241231ddpg.csv')
df_daily_return_td3.to_csv('******trade20241231td3.csv')
df_daily_return_sac.to_csv('******trade20241231sac.csv')